## HTTP Layer — `curl`

**Support context:** `nc -zv` proves the **TCP port** accepts connections. It doesn't tell you whether the **app** behind it works. `curl` speaks HTTP, so it answers the next question: **"when my request arrives, what does the app say back?"** This is Step 4 of the ticket checklist. Most "the site is down" tickets turn out to be one of four things: **DNS, connection, TLS, or the app returning an error**. `curl -v` shows you which one in a single command.

**Developer bridge:** curl is `requests.get()` from the shell.
- `-v` is like logging the full request and response.
- `-I` is `requests.head()`.
- `-w` is like wrapping the call in timing code.
- The status codes are the ones your Django/FastAPI apps return behind gunicorn and nginx: 404, 500, 502.

**COMMAND:** `curl`

**PURPOSE**:   Prove the HTTP layer works (status code, headers, TLS, timing)

**SYNTAX:** `curl [flags] URL`

| Flag | Meaning | When you reach for it |
|---|---|---|
| `-v` | Verbose: DNS, connect, TLS, request and response headers | First move on any "site broken" ticket |
| `-s` | Silent: no progress meter | Scripts. `-sv` keeps the verbose lines without the progress meter. |
| `-I` | HEAD request, headers only | Quick status check without the HTML |
| `-i` | Include response headers **and** the body | When you want to see both |
| `-L` | Follow redirects (301/302) | When `-I` shows a 301 and you need the final page |
| `-o /dev/null` | Throw the body away | Paired with `-w` when you only want metrics |
| `-w '...'` | Print metrics after the request | Status code and timing breakdown |
| `-k` | Skip TLS cert verification | **Diagnosis only.** Proves the cert is the problem. |
| `-H 'Host: x'` | Set a request header | Test a vhost by IP without changing DNS |
| `--resolve host:port:IP` | Force a hostname to an IP for this one request | Bypass DNS, like a one-off `/etc/hosts` entry |
| `-m N` / `--connect-timeout N` | Overall / connect-phase time limit | Stop waiting on blackholed hosts (your last ticket) |
| `-f` | Non-zero exit code on HTTP ≥ 400 | Scripts and health checks |

**EXAMPLE 1 — read a verbose request:**
```bash
curl -sv https://example.com -o /dev/null
```
Real output from your machine (certificate-chain and HTTP/2 stream lines trimmed):

In [ ]:
$ curl -sv https://example.com -o /dev/null
* Host example.com:443 was resolved.
* IPv6: 2606:4700:10::ac42:93f3, 2606:4700:10::6814:179a
* IPv4: 172.66.147.243, 104.20.23.154
*   Trying 172.66.147.243:443...
* Connected to example.com (172.66.147.243) port 443
* ALPN: curl offers h2,http/1.1
*  CAfile: /etc/ssl/certs/ca-certificates.crt
* SSL connection using TLSv1.3 / TLS_AES_256_GCM_SHA384 / X25519 / id-ecPublicKey
* ALPN: server accepted h2
* Server certificate:
*  subject: CN=example.com
*  start date: Sep 26 22:49:11 2026 GMT
*  expire date: Dec 25 22:56:35 2026 GMT
*  subjectAltName: host "example.com" matched cert's "example.com"
*  issuer: C=US; O=SSL Corporation; CN=Cloudflare TLS Issuing ECC CA 3
*  SSL certificate verify ok.
> GET / HTTP/2
> Host: example.com
> User-Agent: curl/8.5.0
> Accept: */*
>
< HTTP/2 200
< date: Wed, 11 Nov 2026 19:04:03 GMT
< content-type: text/html; charset=utf-8
< server: cloudflare
< cf-cache-status: HIT
<
* Connection #0 to host example.com left intact

**READ THE OUTPUT:** there are three line prefixes, and each is a layer you can check.

| Prefix | Who wrote it | What it proves |
|---|---|---|
| `*` | curl itself: resolve, connect, TLS | `was resolved` = DNS OK. `Connected to` = TCP OK (what `nc` proved). `SSL certificate verify ok` = TLS OK. **If it fails here, the app never saw your request.** |
| `>` | Your request | Method, path, and `Host:` header. Check `Host:` when testing vhosts. |
| `<` | The server's reply | Status line + headers. **If you got this far, the network is fine. The problem is in the app or the proxy.** |

Lines worth reading on every ticket:
- `expire date` — an expired cert is a classic ticket.
- `subjectAltName ... matched` — if the name doesn't match, curl fails with exit 60.
- `server: cloudflare` + `cf-cache-status` — a CDN sits in front, so the response may never have touched the origin server.

**EXAMPLE 2 — your own server, three outcomes (two terminals):**

Terminal A is the "service":
```bash
mkdir -p ~/curltest && cd ~/curltest && echo "hello from dennie" > index.html
python3 -m http.server 8000 --bind 127.0.0.1
```
Terminal B is the "client":
```bash
curl -i http://localhost:8000/        # 200 + body
curl -I http://localhost:8000/nope    # 404: server UP, path wrong
```

In [ ]:
$ curl -i http://localhost:8000/
HTTP/1.0 200 OK
Server: SimpleHTTP/0.6 Python/3.12.3
Date: Wed, 11 Nov 2026 19:04:04 GMT
Content-type: text/html
Content-Length: 18
Last-Modified: Wed, 11 Nov 2026 19:04:03 GMT

hello from dennie

$ curl -I http://localhost:8000/nope
HTTP/1.0 404 File not found
Server: SimpleHTTP/0.6 Python/3.12.3
Date: Wed, 11 Nov 2026 19:04:04 GMT
Connection: close
Content-Type: text/html;charset=utf-8
Content-Length: 335

Now **Ctrl+C in Terminal A** to stop the server, then in Terminal B:
```bash
curl -sv http://localhost:8000/ ; echo "exit code: $?"
```

In [ ]:
$ curl -sv http://localhost:8000/ ; echo "exit code: $?"
* Host localhost:8000 was resolved.
* IPv6: ::1
* IPv4: 127.0.0.1
*   Trying [::1]:8000...
* connect to ::1 port 8000 from ::1 port 41496 failed: Connection refused
*   Trying 127.0.0.1:8000...
* connect to 127.0.0.1 port 8000 from 127.0.0.1 port 40042 failed: Connection refused
* Failed to connect to localhost port 8000 after 0 ms: Couldn't connect to server
* Closing connection
exit code: 7

**READ THE OUTPUT:**
- **404 vs refused.** The 404 has `<` lines: the server answered and said "no such path". The refused run has **no `<` lines at all**: nothing was listening.
- **`after 0 ms`** — refused is instant. A firewall drop (your last ticket) hangs until the timeout. Same rule as `nc`: refused is fast, a timeout is slow.
- **`localhost` tried `::1` first, then `127.0.0.1`.** `localhost` resolves to both. The server was bound to `127.0.0.1` only, so even while it was running, the `::1` attempt would be refused and curl would fall back to IPv4. In a real ticket, "works with `127.0.0.1`, fails with `localhost`" points straight at this.
- **Exit code 7** = couldn't connect. That maps to the `ss -tlnp` lesson: what is listening, and on which address.

**EXAMPLE 3 — timing breakdown (where is it slow?):**
```bash
curl -s -o /dev/null -w 'code=%{http_code} dns=%{time_namelookup} connect=%{time_connect} tls=%{time_appconnect} ttfb=%{time_starttransfer} total=%{time_total}\n' https://example.com
```

In [ ]:
$ curl -s -o /dev/null -w 'code=%{http_code} dns=%{time_namelookup} connect=%{time_connect} tls=%{time_appconnect} ttfb=%{time_starttransfer} total=%{time_total}\n' https://example.com
code=200 dns=0.060713 connect=0.094789 tls=0.149605 ttfb=0.225108 total=0.225214

**READ THE OUTPUT:** each value is a **timestamp since start**, not a duration. Subtract to get each phase:

| Phase | Calculation | Your run | Big value means |
|---|---|---|---|
| DNS | `dns` | 61 ms | Resolver problem → `dig`, `/etc/resolv.conf` |
| TCP connect | `connect - dns` | 34 ms | Network latency or packet loss |
| TLS handshake | `tls - connect` | 55 ms | Slow TLS, often a distant server or an overloaded LB |
| App thinking | `ttfb - tls` | 75 ms | **The app is slow**: a slow DB query, gunicorn workers saturated |
| Download | `total - ttfb` | ~0 ms | Big body or slow link |

A server-side error, for comparison:

In [ ]:
$ curl -s -o /dev/null -w '%{http_code}\n' https://httpbin.org/status/503
503

**curl exit codes = the layer that failed.** Memorize these. They tell you where it broke before you've read a single line of `-v` output.

| Exit | Meaning | Layer → next command |
|---|---|---|
| 6 | Couldn't resolve host | DNS → `dig`, `/etc/resolv.conf` |
| 7 | Couldn't connect (refused) | Nothing listening → `ss -tlnp` on the server |
| 28 | Timeout | Firewall drop / routing → `ping`, `traceroute`, ufw |
| 35 / 60 | TLS handshake / cert verify failed | Cert expired, wrong name, missing CA |
| 52 | Empty reply from server | App crashed mid-request |
| 56 | Connection reset | Proxy/LB killed the connection |
| 0 + `4xx`/`5xx` | HTTP worked, the app said no | App logs → `journalctl -u nginx` |

**Key point:** exit code **0 with a 500** is still a success from curl's point of view, because the HTTP exchange completed. Add `-f` if a script needs to treat that as a failure.

Exit 6 on your machine:

In [ ]:
$ curl -s https://nosuchhost.invalid; echo "exit: $?"
exit: 6

**HTTP status codes you'll meet on tickets:**

| Code | Meaning | Support read |
|---|---|---|
| 200 | OK | Works from here. Ask what the user sees differently (path, auth, location, cache). |
| 301 / 302 | Redirect | Check `location:`. A loop shows the same URL repeating under `-L -v`. |
| 401 / 403 | Unauthorized / Forbidden | Auth or permissions, not an outage |
| 404 | Not found | Server up, wrong path or vhost |
| 500 | Internal server error | App exception → app logs |
| 502 | Bad gateway | nginx is up, the **backend** (gunicorn) isn't answering |
| 503 | Service unavailable | Overloaded, or in maintenance |
| 504 | Gateway timeout | Backend reachable but too slow |

**TICKET SCENARIO:** *"Customers get an error page on https://app.acme.io, but our monitoring says the server is up."*

**DIAGNOSIS PATH:**
1. `curl -sI https://app.acme.io`. **502** → nginx up, backend down. **503** → overloaded or maintenance. **504** → backend too slow.
2. `curl -sv https://app.acme.io -o /dev/null`. Does TLS succeed? Does the `server:` header match what you expect, or is there a CDN or LB in front?
3. On the server: `curl -I http://127.0.0.1:8000` hits gunicorn directly and **skips nginx**. If it returns 200, the problem is in the nginx config or the upstream address.
4. `curl --resolve app.acme.io:443:10.0.0.12 https://app.acme.io` tests one specific backend node, bypassing DNS and the LB.

**WHAT TO LOOK FOR:**
- `* Trying ...` with nothing after it → timeout (network or firewall)
- `Connection refused` → service down, or bound to a different address
- `SSL certificate problem` → cert issue. Confirm with `-k`, but don't ship `-k` as the fix.
- `< HTTP/1.1 301` + `location:` → follow with `-L -v` and watch for a loop

**GOTCHAS**
- **`-k` is a diagnostic, not a fix.** If `-k` works, you've proven the cert is the problem. Now go fix the cert.
- **Proxy env vars.** `env | grep -i proxy`. A stale `https_proxy` exported in `.bashrc` makes curl fail while the browser works, which is common on corporate laptops. Use `curl --noproxy '*'` to test without the proxy.
- **`-I` sends HEAD, not GET.** Some apps answer HEAD differently (405, or a different code). If the results look odd, use `curl -s -o /dev/null -w '%{http_code}'`, which sends a real GET.
- **`-v` with `-o` shows a progress meter** mixed into the output. Use `-sv` to keep the verbose lines and drop the meter.
- **WSL:** a server bound to `127.0.0.1` inside WSL2 is usually still reachable from Windows at `localhost:8000`, thanks to WSL's localhost forwarding. That's a WSL convenience; on real Linux, `127.0.0.1` means this machine only.
- **WSL / Windows:** in Windows PowerShell 5.1, `curl` is an alias for `Invoke-WebRequest`. Use `curl.exe` there to get the real curl.

### TRY NOW

1. Restart the Python server with `--bind 0.0.0.0` instead of `127.0.0.1`. Run `ss -tlnp | grep 8000` and compare the local address with a `127.0.0.1` run. This previews the `ss` lesson.

Terminal A, to start the server bound to localhost only:

In [ ]:
# terminal A
mkdir -p ~/curltest && cd ~/curltest && echo "hello from dennie" > index.html

python3 -m http.server 8000 --bind 127.0.0.1
Serving HTTP on 127.0.0.1 port 8000 (http://127.0.0.1:8000/) ...

Terminal B, to see what's listening:

In [ ]:
# 1. paste ss -tlnp output for both binds here

# terminal B
ss -tlnp | grep 8000
LISTEN 0      5           127.0.0.1:8000       0.0.0.0:*    users:(("python3",pid=24501,fd=3))

In [ ]:
# terminal A
python3 -m http.server 8000 --bind 0.0.0.0
Serving HTTP on 0.0.0.0 port 8000 (http://0.0.0.0:8000/) ...

In [ ]:
# terminal B
ss -tlnp | grep 8000
LISTEN 0      5             0.0.0.0:8000       0.0.0.0:*    users:(("python3",pid=24979,fd=3))

**Local Address:Port** column. It shows `127.0.0.1:8000` for the first run and `0.0.0.0:8000` for the second.
You don't need sudo here. You can see `python3` in the process column because the server runs as you, `dennie`. For other users' processes you'd need `sudo ss -tlnp`.

2. Find the slow phase. Run the `-w` timing command against `https://httpbin.org/delay/3`. Which phase takes the 3 seconds, and what does that tell you?

In [ ]:
# 2. paste httpbin.org/delay/3 timing output here

curl -s -o /dev/null -w 'code=%{http_code} dns=%{time_namelookup} connect=%{time_connect} tls=%{time_appconnect} ttfb=%{time_starttransfer} total=%{time_total}\n' https://httpbin.org/delay/3
code=200 dns=0.199011 connect=0.346441 tls=0.560324 ttfb=3.691480 total=3.691583

Subtract each value from the one after it, like in the notebook's timing table, and find which gap is about 3 seconds.

3. A blcackholed host. Run `curl -m 3 http://10.255.255.1 ; echo $?`. Which exit code does a blackholed host give you?

In [ ]:
# 3. paste curl -m 3 http://10.255.255.1 output + exit code here

time curl -m 3 http://10.255.255.1 ; echo "exit code: $?"
curl: (28) Connection timed out after 3001 milliseconds

real    0m3.095s
user    0m0.024s
sys     0m0.019s
exit code: 28

Added `time` so you can check that it's a slow failure, like the `real 3.1s` you got in the `nc` lesson. Note which exit code you get and compare it with the 7 from connection refused.

### DRILL

**Q1.** `curl -sv https://shop.example` prints `* Trying 203.0.113.10:443...` and then sits there until `-m` expires. Which exit code do you get, and which layer failed?

**Q2.** `curl -sI https://shop.example` returns `HTTP/1.1 502 Bad Gateway` with `server: nginx`. Is the server down? What do you check next, and on which machine?

**Q3.** curl fails with exit 60, and `curl -k` works. What have you proven, and why is "just add `-k`" the wrong fix to hand the customer?

**Q4.** In your timing output `ttfb - tls` is 4.8 s and every other phase is under 100 ms. Network team or app team?

### DRILL ANSWERS

1. Timeout - Network or Firewall drop, code 28. Routing layer is failed, so `ping`, `traceroute`, ufw is to reach for next.
2. The web server nginx is up, but the backend server (gunicorn) isn't answering. Next is definetaly to check the backend server. I could check first if the backend host is reachable with `ping` and `traceroute`, then ports could be check as well.
3. 60 means cert verify failed that is cert expired, wrong name or missing CA. `-k` is a diagnostic tool, not a fix. If `-k` works, I've only proven the cert is the problem. Fixing the cert is required.
4. `ttfb - tls` 4.8 s means the app is thinking, it is too slow: a slow DB query, gunicorn workers saturated. But every other phase is under 100 ms meanig it gets considerably better, and it could be diagnosed with `traceroute` to ping each router on its way to destination. It helps to chech each hop if the problem is the network, an ISP, a specific hop, or the destination itself.

### DRILL REVIEW

**Q1 ✅ Correct.** Exit 28, and the transport layer failed: the packets went out and nothing came back.
- One correction: don't call it the "routing layer" failing. A timeout can mean a firewall drop, the host being down, *or* a broken route, and you don't know which yet. `ping` and `traceroute` are how you narrow it down.

**Q2 🟡 Partial.** The diagnosis is right: nginx is up and the backend isn't answering. The next step should be different.

❌ `ping` / `traceroute` to the backend first.
✅ Check from **the nginx box**:
```bash
sudo tail -n 20 /var/log/nginx/error.log   # look for: connect() failed (111: Connection refused) while connecting to upstream
curl -I http://127.0.0.1:8000             # hit gunicorn directly, skipping nginx
systemctl status gunicorn                  # is it running?
journalctl -u gunicorn -n 50               # why did it stop?
```
💡 **Why:** you already *got* a response from nginx, so the network path to the server works. The backend is usually on the same host or a nearby one, and nginx's error log tells you *why* the upstream failed (refused = backend down, timed out = backend stuck) without guessing.
> 🔁 **Support rule:** a 502 is a message from the proxy about its upstream. Read the proxy's error log before you touch the network.

**Q3 ✅ Correct.** One addition about why `-k` is the wrong fix for the customer:
- It turns off the check that you're talking to the real server, which leaves the connection open to a man-in-the-middle attack.
- It only fixes *your* curl. Every browser and API client still rejects the bad cert.

**Q4 ❌ Half right.** You correctly read 4.8 s of `ttfb - tls` as the app being slow. Then the reasoning went the wrong way.

❌ "Other phases under 100 ms → diagnose with `traceroute`".
✅ Fast other phases are exactly what **clears the network**: DNS, TCP and TLS all finished in milliseconds over the same path. Send it to the **app team** and investigate the app:
```bash
journalctl -u gunicorn --since "10 min ago"                                   # slow request / timeout warnings
sudo -u postgres psql -c "SELECT pid, now()-query_start AS age, query FROM pg_stat_activity WHERE state='active' ORDER BY age DESC;"   # long-running queries
```
💡 **Why:** `ttfb - tls` is time spent *inside the server* after it received your request. No hop on the network path can make that slower while the connect and TLS phases stay fast.
🔁 **Support rule:** the timing breakdown tells you which team owns the ticket. Only send it to the network team if a network phase (`dns`, `connect`, `tls`) is the slow one.

**Score: 2.5 / 4.** Q1 and Q3 solid, Q2 partial, Q4 wrong conclusion. The common thread in Q2 and Q4 is reaching for network tools after the evidence had already ruled the network out.


## REQUEST FLOW — `curl https://example.com` layer by layer

This follows `curl -sv https://example.com` from the moment you press Enter until the connection closes. Every number below is from your real run in Example 1 and Example 3.

### The two models side by side

| OSI | TCP/IP | What it handles | In this request |
|---|---|---|---|
| 7 Application | **Application** | What the app is asking for | HTTP `GET /`, DNS query |
| 6 Presentation | Application | Encoding and encryption | TLS encryption |
| 5 Session | Application | Keeping a conversation going | TLS session, HTTP/2 stream, keep-alive |
| 4 Transport | **Transport** | Delivery between two ports | TCP to port 443, UDP to port 53 for DNS |
| 3 Network | **Internet** | Getting packets between hosts | IP addresses, routing via `ip route` |
| 2 Data Link | **Link** | Getting a frame to the next hop | MAC addresses, ARP, `eth0` |
| 1 Physical | Link | Bits on the medium | WSL: a virtual Hyper-V adapter, no real cable |

TLS doesn't fit neatly into OSI. People usually place it at layers 5 and 6. In TCP/IP it's simply part of the application layer, because it runs on top of TCP. That's the more useful view for support work.

**What each layer calls its unit of data (PDU):**

| Layer | PDU name | Addressed by |
|---|---|---|
| L7 Application | data / message | URL, hostname |
| L4 Transport | **segment** (TCP) / **datagram** (UDP) | port numbers |
| L3 Network | **packet** | IP addresses |
| L2 Data Link | **frame** | MAC addresses |
| L1 Physical | bits | n/a |

So "the packet was dropped" is an L3 statement, and "the port is closed" is an L4 statement. Using the right word in a ticket tells the next engineer which layer you mean.

### Phase 0: curl parses the URL (L7, on your machine)
```
https://example.com/   →   scheme=https  host=example.com  port=443  path=/
```
- The scheme sets the default port: `http` → 80, `https` → 443. An explicit port (`:8000`) overrides it.
- Before anything touches the network, curl checks the proxy variables (`https_proxy`). If one is set, everything below goes to the proxy instead of example.com.
- **Shows in curl -v:** nothing yet.

### Phase 1: DNS, to turn the name into an IP (L7, carried over UDP/53)
```
curl → getaddrinfo()                (your system's resolver library)
     → /etc/nsswitch.conf           says "files dns": check the files first, then DNS
     → /etc/hosts                   any override? (no)
     → /etc/resolv.conf             nameserver 10.255.255.254  (WSL's DNS proxy)
     → WSL proxy → Windows → your ISP's or a public resolver → root → .com → example.com servers
     ← A 172.66.147.243 / AAAA 2606:4700:...
```
- The DNS query itself goes down the whole stack: UDP datagram, then IP packet, then Ethernet frame. So DNS needs layers 2–4 working before it can work at all.
- **Theory:**
  - DNS uses **UDP** because a query and its answer each fit in one small datagram. There's no handshake, so the lookup costs a single round trip.
  - It falls back to **TCP/53** for large answers. That's why `nc -zv 10.255.255.254 53` succeeded in the nc lesson.
  - The **recursive resolver** does the root → TLD → authoritative walk on your behalf and caches the result for the record's TTL. That's why a second lookup is usually much faster.
- **curl -v:** `Host example.com:443 was resolved. IPv4: 172.66.147.243`
- **Timer:** `time_namelookup`. Yours was **61 ms**.
- **If it fails:** exit **6**. Next: `dig`, `/etc/resolv.conf`.

### Phase 2: route and next hop (L3 → L2, done by the kernel)
```
kernel: where does 172.66.147.243 go?
  ip route   → "default via 172.x.x.1 dev eth0"   (the WSL virtual gateway)
  ARP        → "who has 172.x.x.1?" → MAC address of the gateway
```
- Your machine never talks directly to example.com at layer 2. It only ever sends frames to the **next hop's** MAC address. Each router along the path removes the frame and builds a new one. The IP addresses stay the same end to end.
- **Theory:**
  - The kernel picks the route by **longest prefix match**: the most specific matching route wins, and `default` (`0.0.0.0/0`) is the last resort. This is the routing-table material from your subnetting notebooks.
  - **ARP** is a broadcast on the local segment, "who has this IP?". The answer is cached in the neighbour table (`ip neigh`).
  - Each router **decrements TTL by 1**. `traceroute` uses exactly this, by sending packets with TTL 1, 2, 3 and so on.
- **curl -v:** this is invisible. curl hands the work to the kernel.
- **If it fails:** `Network is unreachable` with no route. You saw this with IPv6 in the nc lesson.

### Phase 3: TCP three-way handshake (L4)
```
you  ──SYN──────────────►  172.66.147.243:443    "I want to connect, my seq=x"
you  ◄──────────SYN-ACK──                         "OK, my seq=y, ack=x+1"
you  ──ACK──────────────►                         "ack=y+1": connection established
```
- This takes **one round trip**. It's exactly what `nc -zv` tests.
- **Theory:**
  - **Why three steps?** Each side has to pick an initial sequence number *and* see it acknowledged. That's what gives TCP reliable, ordered delivery.
  - **Your source port** (`40xxx`) is an ephemeral port picked by the kernel. Source IP, source port, destination IP and destination port together (the **4-tuple**) identify this one connection.
  - **RST vs silence:** a host with nothing listening replies **RST**, which is fast. A firewall set to DROP replies nothing, so the client keeps retrying its SYN until it gives up. That's the refused-vs-timeout rule.
- **curl -v:** `Trying 172.66.147.243:443...` then `Connected to example.com`
- **Timer:** `time_connect`. You measured **34 ms** after DNS, which is roughly one round trip to Cloudflare's Frankfurt site (`cf-ray: ...-FRA`).
- **If it fails:**
  - A RST comes back: **refused**, exit **7**. Next: `ss -tlnp` on the server.
  - Nothing comes back: **timeout**, exit **28**. Next: firewall or route.

### Phase 4: TLS handshake (OSI L5/6, TCP/IP application layer)
```
you  ──ClientHello──────►   SNI=example.com, ALPN=h2,http/1.1, key share (X25519)
you  ◄──ServerHello──────   chosen cipher, key share, certificate (encrypted in TLS 1.3)
     [curl verifies]         chain → /etc/ssl/certs CA?  name matches SAN?  not expired?
you  ──Finished─────────►   both sides derive the session keys, and the encrypted channel is up
```
- In TLS 1.3 this takes **one round trip** on top of TCP.
- **SNI** tells the server which site's certificate to send. That's how one IP can serve thousands of HTTPS sites.
- **ALPN** agrees on the HTTP version, here `server accepted h2`.
- **Theory: two separate jobs.**
  1. **Key exchange** (`X25519`, Diffie-Hellman): both sides compute the same secret without ever sending it. That secret encrypts everything that follows (`TLS_AES_256_GCM_SHA384`).
  2. **Authentication** (the certificate): proves you are talking to the real `example.com`. The cert is signed by an intermediate CA, which is signed by a root CA that your system trusts (in `/etc/ssl/certs`). That's the `Certificate level 0…3` chain in your output.
  - `-k` skips job 2 only. The connection is still encrypted, but you no longer know *who* is on the other end. That's why it's a diagnostic, not a fix.
  - **TLS 1.2** needed two round trips. **TLS 1.3** needs one.
- **curl -v:** `SSL connection using TLSv1.3`, `subject: CN=example.com`, `expire date`, `SSL certificate verify ok`
- **Timer:** `time_appconnect`. Yours was **55 ms** after connect.
- **If it fails:**
  - exit **35**: the handshake itself failed (no shared protocol, server reset).
  - exit **60**: certificate verification failed (expired, wrong name, unknown CA). `-k` confirms that's the cause.

### Phase 5: the HTTP request goes out (L7, wrapped in every layer below)
```
> GET / HTTP/2
> Host: example.com
```
On the way out, each layer wraps the one above it (**encapsulation**):
```
┌ Ethernet frame (L2)   dst MAC = gateway's MAC ─────────────────────────────┐
│ ┌ IP packet (L3)      src 172.x.x.x → dst 172.66.147.243 ─────────────────┐ │
│ │ ┌ TCP segment (L4)  src port 40xxx → dst port 443, seq/ack ──────────┐ │ │
│ │ │ ┌ TLS record      encrypted ─────────────────────────────────────┐ │ │ │
│ │ │ │  HTTP/2:  GET / , Host: example.com                            │ │ │ │
│ │ │ └────────────────────────────────────────────────────────────────┘ │ │ │
│ │ └─────────────────────────────────────────────────────────────────────┘ │ │
│ └─────────────────────────────────────────────────────────────────────────┘ │
└─────────────────────────────────────────────────────────────────────────────┘
```
- Anyone watching the network in between can see the IP addresses, the ports, and the SNI name. They **cannot** see the path, headers or body.
- **Theory:**
  - **HTTP/1.1** sends text requests one at a time per connection.
  - **HTTP/2** is binary and **multiplexes** many requests as streams over one TCP connection. That's the `[HTTP/2] [1] OPENED stream` line, where `[1]` is the stream ID.
  - **HTTP/3** replaces TCP with **QUIC over UDP**. The `alt-svc: h3=":443"` header in your output was Cloudflare offering HTTP/3 for next time.

### Phase 6: the server side (outside your view)
```
Cloudflare edge (CDN) ──cache HIT?──► answers straight away   (cf-cache-status: HIT)
              └─ miss ──► origin LB → nginx → gunicorn → Django/FastAPI → PostgreSQL
```
- The server takes the layers off in reverse order (**decapsulation**, L2 → L7) and runs the app.
- **Theory:** with a CDN or reverse proxy in front, **your TCP and TLS connections end at the edge**. The edge opens its own, separate connection to the origin. That's why a cert error or a 502 can come from the proxy rather than the app, and why the `server:` header matters.
- **Timer:** `time_starttransfer` (TTFB). Your `ttfb - tls` was **75 ms**. This is the server-side time from Q4: if this gap is large, the app team owns the ticket.

### Phase 7: the response comes back (L7 → L1 → L7)
```
< HTTP/2 200
< server: cloudflare
< content-type: text/html
<
[body]
```
- **Theory:** the status line is the app's verdict. The **first digit is the class**: 2xx success, 3xx redirect, 4xx the client's fault, 5xx the server's fault. TCP ACKs flow back for every chunk of the body, so a lossy link shows up here as a slow `total - ttfb`.
- **Timer:** `time_total`.
- **If it fails:**
  - exit **52**: the server closed the connection without sending anything (app crashed).
  - exit **56**: the connection was reset partway through.
  - **exit 0 with a 4xx or 5xx code**: the network worked fine and the app returned an error.

### Phase 8: close or reuse (L4)
- **curl -v:** `Connection #0 to host example.com left intact`
- The TCP connection stays open (keep-alive), so a browser can reuse it without repeating Phases 1–4. When it does close, each side sends a FIN and the other ACKs it (a four-step close).
- **Theory:** the side that closes first sits in **TIME_WAIT** for a while, so stray late packets can't confuse a new connection. Thousands of TIME_WAIT entries in `ss -s` on a busy server are normal, not a leak.

### The whole thing on one timeline

```
 t=0       61ms         95ms          150ms           225ms     225ms
  │─ DNS ───│── TCP ─────│── TLS ──────│── server ─────│─ body ─│
  │ UDP/53  │ SYN/SYNACK │ Hello/cert  │ app thinking  │        │
  │ exit 6  │ exit 7/28  │ exit 35/60  │ 5xx / exit 52 │ exit 56│
  namelookup  connect      appconnect    starttransfer    total
     L7         L4          L5–6 (TLS)       L7            L7
```

**Round-trip cost of a fresh HTTPS request:** DNS (1 RTT, if not cached) + TCP (1 RTT) + TLS 1.3 (1 RTT) + HTTP request/response (1 RTT) = **about 4 round trips** before the first byte. That's why latency (distance) hurts HTTPS much more than bandwidth does, and why CDNs put edges close to users.

**The support mental model:** each phase only starts once the previous one has worked. A failure tells you that every earlier phase **passed**. Exit 60, for example, means DNS, routing and TCP were all fine, so don't spend time on `ping`. This is the same lesson as Q2 and Q4 in the drill.

🔁 **Support rule:** find the **last phase that succeeded** in `curl -v`. The problem lives in the next one.

### TRY NOW: see the hidden phases

curl doesn't show Phase 2, so look at it directly:

```bash
ip route get 172.66.147.243      # which route and interface the kernel picks (Phase 2, L3)
ip neigh                         # the ARP table: gateway IP → MAC (Phase 2, L2)
curl -sv https://example.com -o /dev/null 2>&1 | grep -E 'resolved|Trying|Connected|SSL connection|verify|^> GET|^< HTTP'
```

The `grep` cuts the verbose output down to one line per phase. Check that you can match each line to Phases 1–7.

**WSL note:** the gateway in `ip neigh` is the Hyper-V virtual switch on your Windows host, not your home router. Your packets go through WSL's NAT, then out through Windows' real network adapter. Pillar 6 finishes with the WSL networking deep dive, which covers that.

In [ ]:
# paste ip route get 172.66.147.243 output here

In [ ]:
# paste ip neigh output here

In [ ]:
# paste the grep'd curl -sv output here, and label each line with its phase

**Theory follow-up (after the hands-on above):**

| Topic | Where | Why it's worth the watch |
|---|---|---|
| HTTP request/response | freeCodeCamp **"FTP, HTTP, ARP & ICMP" — 12:01:21** | The model behind the `>` and `<` lines. It also covers ARP (Phase 2). |
| Layered model | freeCodeCamp **"OSI Model & Layered Architecture" — 28:36** | Fills in the 7-layer map used in the Request Flow section |
| TCP handshake | freeCodeCamp **"TCP Flags & Handshake" — 9:40:48** | SYN/ACK/RST/FIN flags in depth (Phases 3 and 8) |
| TCP header | freeCodeCamp **"TCP Header: Sequence & Ack" — 9:18:10** | What the seq/ack numbers in the handshake actually count |
| TTL | freeCodeCamp **"TTL & Fragmentation" — 8:35:39** | The TTL decrement that makes `traceroute` work (Phase 2) |
| DNS | freeCodeCamp **"DNS Hierarchy & Queries" — 11:57:10** | The root → TLD → authoritative walk (Phase 1) |
| TCP vs UDP | freeCodeCamp **"TCP/UDP Timers & Comparison" — 10:13:30** | Why DNS uses UDP while HTTP/1.1 and HTTP/2 use TCP |
| SSL/TLS | Search **PowerCert** for "SSL TLS" | Short animated explainer of the handshake and certificates (Phase 4) |
| HTTPS in practice | Search **NetworkChuck** for "HTTPS" or "SSL" | Practical, hands-on angle on certs |